# Supervised Learning Problem Types

This notebook explains how to identify supervised learning problems before training a model. The goal is to separate classification from regression, recognize common subtypes, and choose evaluation metrics that match the business problem.

## Why problem type matters

Training starts with a target variable. If the target is a category, the task is classification. If the target is a continuous value, the task is regression. That one decision changes the algorithms you can use, the loss function you optimize, and the metrics you report.

Training a model before answering that question is a design error, not just a missing step.

## Supervised learning in one line

Supervised learning uses labeled examples: input features `X` and correct answers `y`. The model learns a mapping from inputs to outputs and then applies that mapping to unseen data.

$X + y 
ightarrow training 
ightarrow model 
ightarrow predictions$

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import accuracy_score, mean_absolute_error, classification_report

pd.set_option("display.max_columns", 10)

## Classification

Classification predicts a discrete label. The output is a category, not a number on a continuous scale.

Common variants:
- Binary classification: two classes
- Multi-class classification: exactly one class from three or more choices
- Multi-label classification: multiple classes can apply at the same time

In [ ]:
binary_X, binary_y = make_classification(
    n_samples=12,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    random_state=42,
)

binary_df = pd.DataFrame(binary_X, columns=["feature_1", "feature_2", "feature_3", "feature_4"])
binary_df["target"] = binary_y
binary_df.head()

In [ ]:
multi_class_y = pd.Series(["setosa", "versicolor", "virginica", "setosa", "virginica"])
multi_label_y = pd.DataFrame({
    "action": [1, 0, 1],
    "comedy": [0, 1, 1],
    "drama": [1, 1, 0],
})

print("Binary target values:", sorted(binary_df['target'].unique()))
print("Multi-class target values:", sorted(multi_class_y.unique()))
print("Multi-label target matrix:")
display(multi_label_y)

## Regression

Regression predicts a continuous value. The output is a number with magnitude, so prediction error is measured by distance from the true value.

Common variants:
- Standard regression: continuous numeric targets
- Count regression: non-negative integer targets
- Bounded regression: values constrained to a range such as percentages

In [ ]:
regression_X, regression_y = make_regression(
    n_samples=12,
    n_features=4,
    noise=15.0,
    random_state=42,
)

regression_df = pd.DataFrame(regression_X, columns=["feature_1", "feature_2", "feature_3", "feature_4"])
regression_df["target"] = regression_y
regression_df.head()

In [ ]:
y_true_class = np.array([0, 1, 1, 0, 1])
y_pred_class = np.array([0, 1, 0, 0, 1])

y_true_reg = np.array([100.0, 120.0, 90.0, 150.0])
y_pred_reg = np.array([110.0, 115.0, 95.0, 140.0])

classification_accuracy = accuracy_score(y_true_class, y_pred_class)
regression_mae = mean_absolute_error(y_true_reg, y_pred_reg)

print(f"Classification accuracy: {classification_accuracy:.2f}")
print(f"Regression MAE: {regression_mae:.2f}")

## How to identify the problem type

Use the target variable and the business requirement to decide the task type.

Ask these questions:
1. Is the output a category or a continuous number?
2. How many possible outcomes are there?
3. Are the outcomes mutually exclusive, or can multiple apply at once?
4. What does success look like for the business?
5. Which metric actually measures that success?

In [ ]:
def infer_problem_type(target_series: pd.Series) -> str:
    unique_values = target_series.dropna().unique()

    if pd.api.types.is_numeric_dtype(target_series):
        if np.allclose(unique_values, unique_values.astype(int)) and len(unique_values) < 20:
            return "Potential count regression or small-cardinality classification, check semantics"
        return "Regression"

    if len(unique_values) == 2:
        return "Binary classification"

    return "Classification"

print("Binary target:", infer_problem_type(binary_df['target']))
print("Multi-class target:", infer_problem_type(multi_class_y))
print("Regression target:", infer_problem_type(regression_df['target']))

## Practical takeaway

Training produces artifacts. Inference consumes artifacts.

Before you write code, define the problem type clearly. If the target is a category, use classification. If the target is a continuous value, use regression. The rest of the pipeline follows from that decision.